# Week 3: Gaussian-process emulator for ClimateBench

Goal: learn a map from forcing (greenhouse gases and aerosol emissions) to annual-mean climate fields, then score it on the
held-out `ssp245` scenario over 2080–2100.

Run with the **Python (jcm-gp)** kernel (`environment-gp.yml`).

| | |
|---|---|
| **Train** | historical, hist-GHG, hist-aer, ssp126, ssp370, ssp585 |
| **Test** | ssp245 (never used for fitting or normalising anything) |
| **Inputs per year** | CO2, CH4, and the leading 5 principal components (EOFs) of the SO2 and BC emission maps |
| **Targets** | tas, diurnal_temperature_range, pr, pr90: one GP per target |

In [ ]:
import os
import time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's startup log

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import gpflow

DATA = os.path.expanduser("~/data/climatebench")
FIGS = os.path.expanduser("~/DSC180-D06-JCM-Project/figures")
os.makedirs(FIGS, exist_ok=True)

TRAIN = ["historical", "hist-GHG", "hist-aer", "ssp126", "ssp370", "ssp585"]
TEST = "ssp245"
TARGETS = ["tas", "diurnal_temperature_range", "pr", "pr90"]
N_PCS = 5          # per aerosol species, as the brief specifies
PERIOD = slice(2080, 2100)

assert TEST not in TRAIN

## 1. Load the data

- Inputs: CO2 (cumulative emissions) and CH4 are one number per year; SO2 and BC are emission maps (Week 1, Task 3).
- Outputs: each experiment has 1–3 ensemble members. We train on the **ensemble mean**, which averages out some internal
  variability so the GP sees more of the forced response.
- Precipitation is converted from kg m⁻² s⁻¹ to mm/day (× 86400), as in Week 1.

In [ ]:
def load_inputs(exp):
    return xr.open_dataset(f"{DATA}/inputs_{exp}.nc")


def load_outputs(exp):
    ds = xr.open_dataset(f"{DATA}/outputs_{exp}.nc")[TARGETS]
    ds["pr"] = ds["pr"] * 86400
    ds["pr90"] = ds["pr90"] * 86400
    return ds


inputs = {exp: load_inputs(exp) for exp in TRAIN + [TEST]}
outputs = {exp: load_outputs(exp) for exp in TRAIN + [TEST]}

for exp in TRAIN + [TEST]:
    print(f"{exp:10s} years {int(inputs[exp].time[0])}-{int(inputs[exp].time[-1])}, "
          f"members {outputs[exp].sizes['member']}")

## 2. Reduce the aerosol maps to principal components

A GP needs a short input vector, so each SO2 and BC map (96 × 144 values) is compressed to 5 numbers per year with PCA:

1. Weight each grid cell by √cos(latitude), so squared values (the variance PCA maximises) are area-weighted. Otherwise the
   many small high-latitude cells would count as much as large tropical ones.
2. Subtract the **training** mean and take an SVD of the stacked training years.
3. A year's PCs are its projection onto the first 5 components, divided by the training std of each PC.

The PCA is fit on the training years only. `ssp245` is projected onto the same components, so the test scenario never shapes the features.

In [ ]:
def fit_pca(fields, n=N_PCS):
    """fields: training maps stacked along time, shape (time, latitude, longitude)."""
    w = np.sqrt(np.cos(np.deg2rad(fields.latitude.values)))
    w = np.repeat(w, fields.sizes["longitude"])            # one weight per flattened grid cell
    X = fields.values.reshape(fields.sizes["time"], -1) * w
    mean = X.mean(axis=0)
    _, s, vt = np.linalg.svd(X - mean, full_matrices=False)
    components = vt[:n]
    scale = ((X - mean) @ components.T).std(axis=0)
    explained = s[:n] ** 2 / np.sum(s ** 2)
    return {"w": w, "mean": mean, "components": components, "scale": scale, "explained": explained}


def project(pca, fields):
    X = fields.values.reshape(fields.sizes["time"], -1) * pca["w"]
    return (X - pca["mean"]) @ pca["components"].T / pca["scale"]


pcas = {}
for name in ["SO2", "BC"]:
    stacked = xr.concat([inputs[exp][name] for exp in TRAIN], dim="time")
    pcas[name] = fit_pca(stacked)
    print(f"{name}: {N_PCS} PCs explain {pcas[name]['explained'].sum():.1%} of training variance "
          f"{np.round(pcas[name]['explained'], 3)}")

## 3. Build the feature table

12 features per year: CO2, CH4, SO2_0–4, BC_0–4. Each column is then standardised with the **training** mean and std.
The GP lengthscales are then on comparable scales, and all start at 1.

In [ ]:
def features(exp):
    cols = {"CO2": inputs[exp]["CO2"].values, "CH4": inputs[exp]["CH4"].values}
    for name in ["SO2", "BC"]:
        pcs = project(pcas[name], inputs[exp][name])
        for i in range(N_PCS):
            cols[f"{name}_{i}"] = pcs[:, i]
    return pd.DataFrame(cols, index=inputs[exp].time.values)


X_train = pd.concat({exp: features(exp) for exp in TRAIN}, names=["experiment", "year"])
X_test = features(TEST)

x_mean, x_std = X_train.mean(), X_train.std()
X_train = (X_train - x_mean) / x_std
X_test = (X_test - x_mean) / x_std
FEATURES = list(X_train.columns)

print("X_train", X_train.shape, "| X_test", X_test.shape)

# a GP reverts to its mean far from the training data: check ssp245 sits inside the training range
outside = ((X_test < X_train.min()) | (X_test > X_train.max())).mean()
print("fraction of ssp245 years outside the training range, per feature:",
      outside[outside > 0].round(2).to_dict() or "none")

## 4. Targets

Each target is a (years × grid cells) matrix of ensemble means. Each grid cell is standardised with its own training mean and
std, so cells with large variability (e.g. tropical rainfall) don't dominate the fit.

In [ ]:
N_CELLS = 96 * 144

# float64: the files store float32, and GPflow works in float64
Y_train = {var: np.concatenate([outputs[exp][var].mean("member").values.reshape(-1, N_CELLS) for exp in TRAIN])
              .astype(np.float64)
           for var in TARGETS}
truth = outputs[TEST].mean("member")       # ssp245 ensemble mean: what we score against
assert all(len(y) == len(X_train) for y in Y_train.values())

y_mean = {var: Y_train[var].mean(axis=0) for var in TARGETS}
y_std = {var: np.where(Y_train[var].std(axis=0) > 0, Y_train[var].std(axis=0), 1.0) for var in TARGETS}

## 5. The GP

**Kernel.** Two parts, added together:

- a **linear kernel on CO2**: Week 1 showed global warming is close to linear in cumulative CO2, so this term carries the
  main trend and lets the model extrapolate sensibly along it;
- a **Matérn-5/2 kernel on all 12 features** with one lengthscale per feature (ARD), for the nonlinear and aerosol
  effects. A short fitted lengthscale means the output is sensitive to that feature.

Plus a constant mean and Gaussian noise. The hyperparameters are fit by maximising the marginal likelihood of the training data
(L-BFGS), so no validation set is needed and `ssp245` is never used.

**One GP per target.** Every grid cell is an output column with the **same** kernel and noise hyperparameters. That keeps
the fit cheap: one 753 × 753 kernel matrix serves all 13,824 cells.

In [ ]:
def make_kernel():
    linear_co2 = gpflow.kernels.Linear(active_dims=[FEATURES.index("CO2")])
    matern = gpflow.kernels.Matern52(lengthscales=np.ones(len(FEATURES)))
    return linear_co2 + matern


def fit_gp(var):
    y = (Y_train[var] - y_mean[var]) / y_std[var]
    model = gpflow.models.GPR(data=(X_train.values, y),
                              kernel=make_kernel(),
                              mean_function=gpflow.mean_functions.Constant(),
                              noise_variance=0.1)
    start = time.time()
    result = gpflow.optimizers.Scipy().minimize(model.training_loss, model.trainable_variables,
                                                options={"maxiter": 1000})
    print(f"{var}: converged={result.success}, iterations={result.nit}, {time.time() - start:.0f}s")
    return model


def predict(model, var):
    # predict_y includes the noise variance: the spread we'd expect around the truth
    mean, variance = model.predict_y(X_test.values)
    mean = mean.numpy() * y_std[var] + y_mean[var]
    std = np.sqrt(variance.numpy()) * y_std[var]
    as_map = lambda a: xr.DataArray(a.reshape(-1, 96, 144), dims=truth[var].dims, coords=truth[var].coords)
    return as_map(mean), as_map(std)


models, pred, pred_std = {}, {}, {}
for var in TARGETS:
    models[var] = fit_gp(var)
    pred[var], pred_std[var] = predict(models[var], var)

**What did each GP learn?** Kernel variances and the Matérn lengthscale per feature (shorter = more important).

In [ ]:
def learned(model):
    linear, matern = model.kernel.kernels
    row = {"linear CO2 var": float(linear.variance.numpy()),
           "matern var": float(matern.variance.numpy()),
           "noise var": float(model.likelihood.variance.numpy())}
    row.update({f"ℓ {f}": float(l) for f, l in zip(FEATURES, matern.lengthscales.numpy())})
    return row


pd.DataFrame({var: learned(m) for var, m in models.items()}).round(3)

## 6. Scores

All over 2080–2100, against the `ssp245` ensemble mean, with area weights ∝ cos(latitude):

- **Spatial NRMSE:** average each field over 2080–2100 first, then take the area-weighted RMSE between the two maps.
  Measures how well the *pattern* is captured.
- **Global NRMSE:** take the area-weighted global mean each year, then the RMSE between the two yearly series.
  Measures the *global trajectory*.
- Both are divided by |global mean of the truth over 2080–2100|, so targets with different units are comparable.
- **Total = spatial + 5 × global.** The global term is small for most targets, and getting the global-mean trajectory right
  matters most for many uses, so it is weighted up.

Note the normalisation: DTR's global-mean change is close to zero, so dividing by it inflates DTR's NRMSE a lot.

In [ ]:
def global_mean(da):
    return da.weighted(np.cos(np.deg2rad(da.lat))).mean(["lat", "lon"])


def nrmse(truth_da, pred_da):
    t, p = truth_da.sel(time=PERIOD), pred_da.sel(time=PERIOD)
    norm = abs(float(global_mean(t.mean("time"))))
    weights = np.cos(np.deg2rad(t.lat))
    spatial = float(np.sqrt(((t.mean("time") - p.mean("time")) ** 2).weighted(weights).mean(["lat", "lon"]))) / norm
    glob = float(np.sqrt(((global_mean(t) - global_mean(p)) ** 2).mean("time"))) / norm
    return {"Spatial": spatial, "Global": glob, "Total": spatial + 5 * glob}


def to_row(scores):
    return pd.Series({(var, k): v for var, d in scores.items() for k, v in d.items()})

**Internal-variability floor (Question 3).** The truth is a 3-member mean, so it still contains weather noise that no emulator
of the forced response can predict. Rough estimate: score each member against the ensemble mean of the other two.
If the members are independent with equal noise, an emulator that is *perfect* at the forced response would score about
√(1/3) / √(1 + 1/2) ≈ 0.47 times that, so we scale by 0.47. Treat this as the order of magnitude of the best achievable NRMSE.

In [ ]:
def variability_floor(var):
    members = outputs[TEST][var]
    scores = []
    for m in members.member.values:
        others = members.drop_sel(member=m).mean("member")
        scores.append(nrmse(others, members.sel(member=m)))
    floor = np.sqrt(1 / 3) / np.sqrt(1 + 1 / 2)
    s = {k: floor * np.mean([d[k] for d in scores]) for k in ["Spatial", "Global"]}
    s["Total"] = s["Spatial"] + 5 * s["Global"]
    return s


# Gaussian-process row of the README leaderboard (github.com/duncanwp/ClimateBench)
published = {
    "tas": {"Spatial": 0.109106, "Global": 0.0738238, "Total": 0.478225},
    "diurnal_temperature_range": {"Spatial": 9.20713, "Global": 2.67495, "Total": 22.5819},
    "pr": {"Spatial": 2.34092, "Global": 0.341453, "Total": 4.04818},
    "pr90": {"Spatial": 2.5559, "Global": 0.429154, "Total": 4.70167},
}

leaderboard = pd.DataFrame({
    "GP (ours)": to_row({var: nrmse(truth[var], pred[var]) for var in TARGETS}),
    "GP (published)": to_row(published),
    "variability floor (approx.)": to_row({var: variability_floor(var) for var in TARGETS}),
}).T
leaderboard.round(3)

## 7. Maps (2080–2100 mean)

Columns: truth, GP mean, **GP − truth**, and the GP's predictive std.

In [ ]:
STYLE = {"tas": ("tas", "K", "RdBu_r"),
         "diurnal_temperature_range": ("dtr", "K", "RdBu_r"),
         "pr": ("pr", "mm/day", "BrBG"),
         "pr90": ("pr90", "mm/day", "BrBG")}

fig, axes = plt.subplots(4, 4, figsize=(22, 14), subplot_kw={"projection": ccrs.Robinson()})
for row, var in enumerate(TARGETS):
    name, unit, cmap = STYLE[var]
    t = truth[var].sel(time=PERIOD).mean("time")
    p = pred[var].sel(time=PERIOD).mean("time")
    s = pred_std[var].sel(time=PERIOD).mean("time")
    lim = float(np.nanpercentile(np.abs(t), 98))
    dlim = float(np.nanpercentile(np.abs(p - t), 98))
    panels = [(t, "truth", dict(cmap=cmap, vmin=-lim, vmax=lim)),
              (p, "GP mean", dict(cmap=cmap, vmin=-lim, vmax=lim)),
              (p - t, "GP − truth", dict(cmap=cmap, vmin=-dlim, vmax=dlim)),
              (s, "GP std", dict(cmap="viridis", vmin=0))]
    for ax, (da, title, kw) in zip(axes[row], panels):
        da.plot(ax=ax, transform=ccrs.PlateCarree(), cbar_kwargs={"label": unit, "shrink": 0.7}, **kw)
        ax.coastlines()
        ax.set_title(f"{name}: {title}")
fig.savefig(f"{FIGS}/week3_gp_maps.png", dpi=150, bbox_inches="tight")

## 8. Is the predictive variance any good?

Two things to keep in mind:

- GP variance depends **only on the inputs**, not on the target values. In standardised units every grid cell gets the same
  variance in a given year; the std map above is that one number times each cell's training std. So the map's *shape*
  comes from the normalisation, and only its *size* comes from the GP.
- If the std is calibrated, the truth should fall within ±2σ of the GP mean about 95% of the time.

The left panel compares the typical predicted σ with the actual RMSE each year; the right panel checks coverage.

In [ ]:
weights = np.cos(np.deg2rad(truth.lat))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
coverage = {}
for var in TARGETS:
    name = STYLE[var][0]
    err = pred[var] - truth[var]
    rmse_t = np.sqrt((err ** 2).weighted(weights).mean(["lat", "lon"]))
    sigma_t = np.sqrt((pred_std[var] ** 2).weighted(weights).mean(["lat", "lon"]))
    axes[0].plot(truth.time, rmse_t / sigma_t, label=name)
    inside = (np.abs(err) < 2 * pred_std[var]).sel(time=PERIOD)
    coverage[name] = float(inside.weighted(weights).mean())

axes[0].axhline(1, color="gray", lw=0.8)
axes[0].set_title("actual RMSE / predicted σ (1 = calibrated)")
axes[0].set_xlabel("year")
axes[0].legend()
axes[1].bar(list(coverage), list(coverage.values()))
axes[1].axhline(0.95, color="gray", ls="--", label="95% (calibrated)")
axes[1].set_ylim(0, 1)
axes[1].set_title("fraction of truth within ±2σ, 2080–2100")
axes[1].legend()
fig.savefig(f"{FIGS}/week3_gp_uncertainty.png", dpi=150, bbox_inches="tight")
print({k: round(v, 3) for k, v in coverage.items()})

## Notes for the write-up

Fill in after running:

- Leaderboard vs published, per target: reproduced or not, and the likely cause.
- Which target is hardest, and why (signal-to-noise; aerosol influence on DTR and precipitation).
- Ensemble spread vs our errors: how close are we to the variability floor?
- What the learned lengthscales say about which forcings matter for each target.